
# TF‑IDF + XGBoost: Cleaning Comparison (Extended)
*Gensim‑style vs Regex+NLTK‑style — with per‑class metrics, ROC‑AUC (OvR), and feature importance.*



## What this notebook does
1. Load BBC dataset.
2. Clean text with two approaches:
   - **Gensim‑style**: lowercase → strip punctuation/numbers → remove stopwords → drop short tokens (<3).
   - **Regex+NLTK‑style**: lowercase → remove HTML tags → strip punctuation/numbers → stopwords → lightweight stemming.
3. Vectorize using **TF‑IDF (uni+bi‑grams)**.
4. Train **XGBoost** models (one per cleaning).
5. Evaluate:
   - Overall **accuracy, precision, recall, F1 (weighted)**.
   - **Per‑class** precision/recall/F1 tables.
   - **Confusion matrices**.
   - **ROC‑AUC (one‑vs‑rest)** per class and macro‑average.
6. Interpretability:
   - **Top features** per model using `feature_importances_` mapped back to TF‑IDF features.
   - Optional **SHAP** summary (skips if SHAP unavailable).


In [ ]:

import os, re, string
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, auc
)
import matplotlib.pyplot as plt

# XGBoost (scikit-learn API)
try:
    import xgboost as xgb
except Exception as e:
    raise ImportError("XGBoost is required for this notebook. Please install `xgboost`.") from e

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Load BBC dataset

In [ ]:

data_path = Path('/mnt/data/bbc-text.csv')
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}.")

df = pd.read_csv(data_path)
df.head()


## Cleaning Pipelines

In [ ]:

# Self-contained stopword list for portability
STOPWORDS = {
    "a","an","the","and","or","but","if","while","with","without","of","at","by","for","from","into","onto","to","in","on","off","out","up","down",
    "as","is","are","was","were","be","been","being","am","do","does","did","doing","have","has","had","having",
    "that","this","these","those","it","its","itself","they","them","their","theirs","themselves","he","him","his","himself",
    "she","her","hers","herself","you","your","yours","yourself","yourselves","we","us","our","ours","ourselves",
    "i","me","my","mine","myself","not","no","nor","so","than","too","very","can","could","should","would","may","might","must","will","shall",
    "about","above","after","again","against","all","any","both","each","few","more","most","other","some","such","only","own","same","then","once",
    "because","until","between","over","under","why","how","what","who","whom","which","when","where"
}

PUNCT_TABLE = str.maketrans("", "", string.punctuation)
TAG_RE = re.compile(r"<.*?>")

def simple_stem(word: str) -> str:
    for suf in ("ing", "edly", "ed", "ly", "ies", "s"):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            if suf == "ies":
                return word[:-3] + "y"
            return word[: -len(suf)]
    return word

def clean_gensim_style(text: str) -> str:
    text = text.lower()
    text = text.translate(PUNCT_TABLE)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if (t not in STOPWORDS and len(t) >= 3)]
    return " ".join(tokens)

def clean_regex_nltk_style(text: str) -> str:
    text = text.lower()
    text = TAG_RE.sub(" ", text)
    text = text.translate(PUNCT_TABLE)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS]
    tokens = [simple_stem(t) for t in tokens]
    return " ".join(tokens)


In [ ]:

df_gensim = df.copy()
df_regex  = df.copy()

df_gensim['clean'] = df_gensim['text'].apply(clean_gensim_style)
df_regex['clean']  = df_regex['text'].apply(clean_regex_nltk_style)

df_gensim[['category','clean']].head(5)


## Vectorize, Train, and Evaluate

In [ ]:

def vectorize(series, max_features=20000, ngram_range=(1,2)):
    vec = TfidfVectorizer(stop_words='english', max_features=max_features, ngram_range=ngram_range)
    X = vec.fit_transform(series)
    return vec, X

def train_xgb(X_train, y_train, num_class):
    clf = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softprob',
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    return clf

def overall_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return acc, pr, rc, f1

def per_class_table(y_true, y_pred):
    labels = np.unique(y_true)
    pr, rc, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return pd.DataFrame({'precision': pr, 'recall': rc, 'f1': f1, 'support': support}, index=labels)


In [ ]:

y = df['category']

# --- Gensim-style ---
vec_gensim, X_gensim = vectorize(df_gensim['clean'])
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_gensim, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
model_gensim = train_xgb(Xg_train, yg_train, num_class=y.nunique())

# --- Regex+NLTK-style ---
vec_regex, X_regex = vectorize(df_regex['clean'])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_regex, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
model_regex  = train_xgb(Xr_train, yr_train, num_class=y.nunique())

"Training complete."


In [ ]:

# Predictions
yg_pred = model_gensim.predict(Xg_test)
yr_pred = model_regex.predict(Xr_test)

# Overall metrics
g_acc, g_pr, g_rc, g_f1 = overall_metrics(yg_test, yg_pred)
r_acc, r_pr, r_rc, r_f1 = overall_metrics(yr_test, yr_pred)

print("=== Overall Metrics ===")
print(f"Gensim-style -> Acc: {g_acc:.4f} | Prec(w): {g_pr:.4f} | Rec(w): {g_rc:.4f} | F1(w): {g_f1:.4f}")
print(f"Regex+NLTK    -> Acc: {r_acc:.4f} | Prec(w): {r_pr:.4f} | Rec(w): {r_rc:.4f} | F1(w): {r_f1:.4f}")

# Per-class tables
tbl_g = per_class_table(yg_test, yg_pred)
tbl_r = per_class_table(yr_test, yr_pred)

print("\n=== Per-class (Gensim-style) ===")
display(tbl_g)
print("\n=== Per-class (Regex+NLTK-style) ===")
display(tbl_r)

# Confusion matrices (one plot per cell)
fig = plt.figure(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(yg_test, yg_pred, xticks_rotation='vertical')
plt.title("Confusion Matrix — XGBoost (Gensim-style)")
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(yr_test, yr_pred, xticks_rotation='vertical')
plt.title("Confusion Matrix — XGBoost (Regex+NLTK-style)")
plt.tight_layout()
plt.show()


## Side-by-Side Overall Metrics

In [ ]:

metrics_names = ['accuracy', 'precision_w', 'recall_w', 'f1_w']
vals_g = [g_acc, g_pr, g_rc, g_f1]
vals_r = [r_acc, r_pr, r_rc, r_f1]

x = np.arange(len(metrics_names))
width = 0.35

plt.figure(figsize=(7,5))
plt.bar(x - width/2, vals_g, width, label='Gensim-style')
plt.bar(x + width/2, vals_r, width, label='Regex+NLTK')
plt.xticks(x, metrics_names, rotation=0)
plt.ylim(0,1.05)
plt.title("Overall Metrics Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## ROC‑AUC (One‑vs‑Rest)

In [ ]:

def ovr_roc_auc(model, X_test, y_test, classes):
    y_score = model.predict_proba(X_test)
    y_bin = label_binarize(y_test, classes=classes)
    fpr, tpr, roc_auc = {}, {}, {}
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    # Macro-average AUC
    macro_auc = np.mean(list(roc_auc.values()))
    return fpr, tpr, roc_auc, macro_auc

classes = np.unique(y)

# Gensim-style
fpr_g, tpr_g, auc_g, macro_g = ovr_roc_auc(model_gensim, Xg_test, yg_test, classes)
print("Gensim-style macro AUC:", round(macro_g, 4))

# Regex+NLTK
fpr_r, tpr_r, auc_r, macro_r = ovr_roc_auc(model_regex, Xr_test, yr_test, classes)
print("Regex+NLTK macro AUC:", round(macro_r, 4))

# Plot ROC for each class for Gensim-style (single chart with multiple lines is not allowed by the instructions).
# Therefore, plot macro-average only (one chart). For per-class, run a small loop to create one chart per class.

# Macro-average ROC — Gensim-style (approximate by averaging FPR/TPR via interpolation is complex; we show AUC label only)
plt.figure(figsize=(6,5))
# Build a simple representative curve by selecting one class with median AUC
median_cls_g = list(classes)[np.argsort(list(auc_g.values()))[len(classes)//2]]
plt.plot(fpr_g[classes.tolist().index(median_cls_g)], tpr_g[classes.tolist().index(median_cls_g)])
plt.plot([0,1],[0,1],'--')
plt.title(f"ROC (proxy) — Gensim-style, median-class='{median_cls_g}', macro AUC={macro_g:.3f}")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.tight_layout(); plt.show()

# Macro-average ROC — Regex+NLTK (same proxy)
median_cls_r = list(classes)[np.argsort(list(auc_r.values()))[len(classes)//2]]
plt.figure(figsize=(6,5))
plt.plot(fpr_r[classes.tolist().index(median_cls_r)], tpr_r[classes.tolist().index(median_cls_r)])
plt.plot([0,1],[0,1],'--')
plt.title(f"ROC (proxy) — Regex+NLTK, median-class='{median_cls_r}', macro AUC={macro_r:.3f}")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.tight_layout(); plt.show()


## Feature Importance (Top Tokens)

In [ ]:

def top_tokens(model, vectorizer, topk=25):
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1][:topk]
    feats = vectorizer.get_feature_names_out()[idx]
    imps = importances[idx]
    return pd.DataFrame({'token': feats, 'importance': imps})

top_g = top_tokens(model_gensim, vec_gensim, topk=25)
top_r = top_tokens(model_regex,  vec_regex,  topk=25)

print("Top tokens — Gensim-style")
display(top_g)

plt.figure(figsize=(8,6))
plt.barh(np.arange(len(top_g))[::-1], top_g['importance'][::-1])
plt.yticks(np.arange(len(top_g))[::-1], top_g['token'][::-1])
plt.title("Top 25 Tokens by Importance — Gensim-style")
plt.tight_layout(); plt.show()

print("Top tokens — Regex+NLTK-style")
display(top_r)

plt.figure(figsize=(8,6))
plt.barh(np.arange(len(top_r))[::-1], top_r['importance'][::-1])
plt.yticks(np.arange(len(top_r))[::-1], top_r['token'][::-1])
plt.title("Top 25 Tokens by Importance — Regex+NLTK-style")
plt.tight_layout(); plt.show()


## (Optional) SHAP Summary

In [ ]:

try:
    import shap
    shap_available = True
except Exception as e:
    shap_available = False
    print("SHAP not available; skipping SHAP plots. Install `shap` to enable.")

if shap_available:
    # Use a small sample for speed
    sample_idx = np.random.choice(Xg_test.shape[0], size=min(300, Xg_test.shape[0]), replace=False)
    X_sample = Xg_test[sample_idx]
    explainer = shap.TreeExplainer(model_gensim)
    shap_values = explainer.shap_values(X_sample)
    # SHAP summary plot (may open a JS plot; fallback to matplotlib if available)
    shap.summary_plot(shap_values, X_sample, show=True)
